# PyTorch Foundations 05 - A working loop isn't an honest experiment

You can deliver sensor rows and update a fuel model. That doesn't stop you
from fitting preprocessing on future evidence, measuring with dropout still
on, or saving the last model when an earlier one was better.

**Guiding question:** Which decisions should stay visible when a trainer
framework eventually automates the loop?

Keep load and temperature as inputs, fuel as the answer. We now add a
curved interaction to the synthetic rule and train a small nonlinear model.
We don't load Part 03's affine weights into this different model.

| Complaint | Smallest repair |
|---|---|
| Future rows help choose the measuring ruler | Split before fitting normalization |
| A repeated prediction changes | Separate model mode from autograd tracking |
| A small remainder gets too much metric weight | Average examples, not batch means |
| Loss stops improving on new rows | Separate scheduling, stopping, and best-model selection |
| A saved model expects forgotten preprocessing | Save and reload the training-derived rulers too |

![Train rows fit preprocessing and update the model; validation measures without gradients and guides scheduling and selection; test evidence stays separate until choices are finished.](../images/05-learning-and-exam-rooms.png)

The setup makes experiment choices visible: batch size 24, rate 0.02,
at most 80 epochs, patience 10. These are settings, not learned parameters.
We'll reload a selected model for test inference and inspect a separate
training snapshot. Exact crash recovery, Lightning, compilation, and
distributed training aren't run here.
[Handwritten theory](05-modern-training-lifecycle-theory.md).


In [ ]:
import sys
print(f"Kernel interpreter: {sys.executable}")
from pathlib import Path
LESSON_DIR = next(
    folder for folder in (
        Path(globals().get("__vsc_ipynb_file__", "")).resolve().parent,
        Path.cwd(),
        Path.cwd().joinpath('learning', '02-pytorch-foundations', '05-modern-training-lifecycle'),
    )
    if (folder / "05-modern-training-lifecycle.ipynb").is_file()
)

# -- Imports, reproducibility, and configuration ------------------------------
from dataclasses import asdict, dataclass
from pathlib import Path
import json
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

torch.manual_seed(53)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

@dataclass
class TrainConfig:
    seed: int = 53
    batch_size: int = 24
    learning_rate: float = 0.02
    max_epochs: int = 80
    patience: int = 10

config = TrainConfig()
print(f"Configuration: {asdict(config)}")


## 1 - Keep future evidence out of the measuring ruler

First make 180 load/temperature rows. The target rule adds an interaction:
the temperature contribution depends on load. A two-weight affine model
can't express that changing contribution by itself.

The next cell makes `[180,2]` inputs and `[180,1]` targets. `unsqueeze(1)`
creates the target column so one fuel prediction can be compared with one
answer; it doesn't add another fuel quantity.

We know the rule because we generated the examples. The model receives
only pairs. Now split those pairs **before** calculating any centers
or scales from them.


In [ ]:
# -- Create a nonlinear fuel-efficiency dataset -------------------------------
generator = torch.Generator().manual_seed(config.seed)
n_samples = 180
engine_load = torch.rand(n_samples, generator=generator)
temperature = 1.8 * torch.rand(n_samples, generator=generator) - 0.8
# Two (180 rows,) sensor vectors -> (180, 2): each row holds load and
# temperature together, in the same feature order as Parts 03–04.
X_raw = torch.stack([engine_load, temperature], dim=1)
# The formula gives (180,) fuel values; unsqueeze -> (180, 1) so each
# target row matches the model's one-column output instead of broadcasting.
y_raw = (
    1.2
    + 1.7 * engine_load
    - 0.8 * temperature
    + 0.55 * engine_load * temperature
    + 0.10 * torch.randn(n_samples, generator=generator)
).unsqueeze(1)
print(f"Raw contracts: X={tuple(X_raw.shape)}, y={tuple(y_raw.shape)}")


### "No labels involved" doesn't mean "no leak"

Normalization chooses one center and scale per sensor column. If validation
and test inputs help choose them, those future inputs have influenced a
training choice even though their fuel labels stayed hidden.

The next block shuffles row addresses into **108 training**, **36 validation**,
and **36 test** rows. Only the 108 choose the ruler. Validation guides
later settings; test waits until those choices finish.

The printed gap between the training-only and all-row means shows which
information an all-row ruler would import. It doesn't measure an accuracy
penalty, and the random split does not deliberately create a shifted
distribution. Read the actual gap rather than assuming a dramatic leak.


In [ ]:
# -- Split first, then fit preprocessing on training rows only ----------------
indices = torch.randperm(n_samples, generator=generator)
train_idx, val_idx, test_idx = indices[:108], indices[108:144], indices[144:]
X_train_raw, X_val_raw, X_test_raw = X_raw[train_idx], X_raw[val_idx], X_raw[test_idx]
y_train, y_val, y_test = y_raw[train_idx], y_raw[val_idx], y_raw[test_idx]

# (108, 2) -> (2,): reduce training rows, keeping one center/scale per sensor.
train_mean = X_train_raw.mean(dim=0)
train_std = X_train_raw.std(dim=0)
# (180, 2) -> (2,) is a deliberately leaked ruler for comparison only.
all_mean = X_raw.mean(dim=0)
# Two feature-center gaps -> scalar maximum: expose any changed sensor ruler.
mean_gap = (train_mean - all_mean).abs().max().item()
print(f"Max train-only versus all-data mean gap: {mean_gap:.4f}")
print("Only train statistics will transform validation and test rows.")


### One ruler, used everywhere

Subtract each training feature mean, then divide by its training standard
deviation. A value then describes distance from typical **training**
measurements. Validation does not get to choose a new zero.

We do the same for fuel targets to make the numerical task convenient.
A predicted normalized fuel value must later be converted back with
`prediction × saved target scale + saved target mean`.

`train_mean` and `train_std` each have two entries, reused by feature
column across rows. The target rulers each have one entry. They are fitted
preprocessing state, not neural-network parameters and not activations
that should be recomputed separately for each split.

The next cell returns normalized train/validation/test tables and prints
their feature means. Training means should be near zero; validation means
are allowed to differ. Erasing that difference would change the question.


In [ ]:
# -- Normalize features and targets without crossing evidence boundaries -------
# (108, 1) -> (1,): fit one fuel center/scale on training targets only.
y_mean = y_train.mean(dim=0)
y_std = y_train.std(dim=0)
# (N, 2) minus/divide by (2,) -> (N, 2): reuse train-only statistics
# by feature column across rows; target stats (1,) likewise keep (N, 1).
normalize_X = lambda values: (values - train_mean) / train_std
normalize_y = lambda values: (values - y_mean) / y_std
X_train, X_val, X_test = map(normalize_X, (X_train_raw, X_val_raw, X_test_raw))
y_train_n, y_val_n, y_test_n = map(normalize_y, (y_train, y_val, y_test))
# (N, 2) -> (2,): inspect one normalized average per feature, not per row.
print(f"Train feature mean after normalization: {X_train.mean(dim=0).tolist()}")
print(f"Validation feature mean remains evidence: {X_val.mean(dim=0).tolist()}")


## 2 - A remainder is small, not less important

The training loader shuffles paired practice rows. Validation and test
loaders don't need shuffle: they measure without updating.

At batch size 24, training delivers `24+24+24+24+12`; each 36-row check
delivers `24+12`. The short group must count by its row count, not as a
full-size vote.

For a pencil example, if those two validation batch means were 1 and 4,
the row-weighted mean would be `(24×1 + 12×4)/36 = 2`, not `(1+4)/2 = 2.5`.
These are example losses, not measurements from this run.

> **Visual to add — Twelve rows don't get twenty-four votes**
> Draw a validation tray with 24 fuel cards and one with 12. Put example
> mean-loss tags 1 and 4 above them. Contrast equal-tray voting, 2.5, with
> per-row voting, 2. Label these as illustrative losses; use no fake
> training curve. Suggested caption: "Multiply batch means by rows before
> averaging the whole split."

The next cell builds the loaders and prints one `[B,2]`/`[B,1]` delivery.
The two loop functions below implement the row-weighted calculation.


In [ ]:
# -- Build loaders with explicit shuffle policies -----------------------------
# Row pairs (N, 2)/(N, 1) -> batches (B, 2)/(B, 1): group examples,
# not columns, while keeping the final smaller batch for fair loss averages.
train_dataset = TensorDataset(X_train, y_train_n)
val_dataset = TensorDataset(X_val, y_val_n)
test_dataset = TensorDataset(X_test, y_test_n)
train_loader = DataLoader(
    train_dataset,
    batch_size=config.batch_size,
    shuffle=True,
    generator=torch.Generator().manual_seed(config.seed),
)
val_loader = DataLoader(val_dataset, batch_size=config.batch_size, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=config.batch_size, shuffle=False)
sample_X, sample_y = next(iter(train_loader))
print(f"Batch contract: X={tuple(sample_X.shape)}, y={tuple(sample_y.shape)}")


## 3 - Same parameters, different layer behavior

The model creates 16 hidden features from two sensor readings, bends them
with ReLU, and mixes them into one normalized fuel prediction. Its layers
own **65 parameters**: 32 first-layer weights, 16 biases, 16 output
weights, and one output bias.

Dropout adds no parameters. In training it randomly sets some hidden
**activations** to zero and rescales retained ones. `Dropout(0.15)` means
15% drop probability, not "delete 15% of the stored weights". The hidden
shape stays `[B,16]`.

During evaluation that masking stops. This is one reason "don't update
weights" isn't a complete description of evaluation behavior.
The next class only defines the guesser and prints its parameter count;
it doesn't normalize data, calculate loss, or start training.


In [ ]:
# -- Define a compact model with mode-sensitive dropout -----------------------
class FuelEfficiencyRegressor(nn.Module):
    def __init__(self):
        super().__init__()
        # Per row: (B, 2 sensors) -> (B, 16 features) -> (B, 1 fuel value).
        # ReLU bends features and dropout masks some during training; neither
        # changes (B, 16), so the final Linear still sees the same feature axis.
        self.network = nn.Sequential(
            nn.Linear(2, 16),
            nn.ReLU(),
            nn.Dropout(0.15),
            nn.Linear(16, 1),
        )

    def forward(self, inputs):
        return self.network(inputs)

model = FuelEfficiencyRegressor().to(device)
print(f"Trainable parameters: {sum(p.numel() for p in model.parameters())}")


### Try the same six rows twice

`model.train()` allows dropout to choose different masks for two otherwise
identical inputs. `model.eval()` disables those masks.
`inference_mode()` separately stops gradient tracking.

> **Visual to add — The weights stay; the temporary routes change**
> Use six fuel rows entering the same 2-to-16-to-1 model in two panels.
> In train mode, show two different temporary masks on the 16 hidden
> activations, labelled drop probability 0.15; keep weight knobs identical.
> In eval mode, show all hidden routes available and identical repeated
> answers. Don't draw made-up prediction values. Suggested caption:
> "Mode changes layer behavior; it doesn't replace learned weights."

**Predict:** Which repeat gap should be exactly zero? The next cell prints
both gaps and asserts evaluation equality. A stochastic training pair can
occasionally match, so it doesn't demand a nonzero gap every time.
This probe trains nothing.


In [ ]:
# -- Measure train/eval behavior on identical input ---------------------------
mode_probe = sample_X[:6].to(device)
model.train()
train_a, train_b = model(mode_probe), model(mode_probe)
model.eval()
with torch.inference_mode():
    eval_a, eval_b = model(mode_probe), model(mode_probe)
# Each (6, 1) repeat difference -> scalar maximum: test the worst repeated guess.
print(f"Train-mode max repeat gap: {(train_a - train_b).abs().max().item():.6f}")
print(f"Eval-mode max repeat gap: {(eval_a - eval_b).abs().max().item():.6f}")
assert torch.equal(eval_a, eval_b)


The forward behavior is clear. Now separate three other jobs:
AdamW chooses parameter updates, a scheduler changes their future rate,
and optional mixed precision changes arithmetic—not the learning target.


## 4 - Give each training tool one job

**AdamW** remembers running gradient summaries and applies weight decay
separately from its gradient-based update. This state belongs to the updater.

**ReduceLROnPlateau** watches validation loss. After enough non-improvement,
it halves the learning rate (`factor=0.5`, scheduler patience 3). It doesn't
backpropagate through validation or decide when the whole experiment stops.

**Mixed precision** lets selected operations use fewer digits on supported
hardware. The gradient scaler enlarges loss for backward, then compensates
before stepping; it can skip an unsafe update. Here both autocast and
scaling are enabled only on CUDA. The printed flag says which path you ran.
A CPU run doesn't establish an accelerator speedup.

The next cell creates these tools. Training begins only when a function
actually calls them in the right order.


In [ ]:
# -- Configure loss, optimizer, scheduler, and mixed-precision boundary -------
loss_function = nn.MSELoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=config.learning_rate, weight_decay=1e-3)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, factor=0.5, patience=3)
use_amp = device.type == "cuda"
scaler = torch.amp.GradScaler("cuda", enabled=use_amp)
print(f"Mixed precision enabled: {use_amp}")
print(f"Initial learning rate: {optimizer.param_groups[0]['lr']}")


### Read one training epoch as input, work, and return value

The function receives a model, loader, optimizer, and scaler. For every
batch it selects training behavior, moves the paired tensors to the device,
clears gradients, predicts under optional autocast, scores, backpropagates,
and steps. `scaler.update()` adjusts numerical scaling for later steps,
not the scheduler's learning rate.

Its return value is one **row-weighted training MSE**. Each batch mean is
multiplied by its actual row count before summing, then divided by 108.
The number mixes predictions made at successive parameter states with
dropout active; it isn't a fresh evaluation of the final model.

Defining the function stores the instructions. It hasn't trained yet.
Next we need a checking function with no permission to practise.


In [ ]:
# -- Define one training epoch without orchestration noise --------------------
def train_one_epoch(model, dataloader, optimizer, scaler):
    model.train()
    total_loss = 0.0
    for batch_X, batch_y in dataloader:
        batch_X, batch_y = batch_X.to(device), batch_y.to(device)
        optimizer.zero_grad()
        with torch.autocast(device_type=device.type, enabled=use_amp):
            loss = loss_function(model(batch_X), batch_y)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        total_loss += loss.item() * len(batch_X)
    return total_loss / len(dataloader.dataset)

print("Defined train_one_epoch: gradients and updates occur only here.")


### Check without quietly practising

`evaluate` receives a model and a loader, selects eval mode, and predicts
inside inference mode. It returns row-weighted MSE with no backward or step.
It deliberately changes the model's mode; the next training call switches
it back.

What about gradients already left from training? Inference doesn't erase
them. The audit therefore copies both parameter values and any `.grad`
buffers **before** evaluation, then checks both afterward. Seeing an old
gradient alone would not prove that validation created one.

The next block defines the function and actually runs this before/after
audit on validation. Read its probe loss as a measurement of the untrained
model, not a final score.


In [ ]:
# -- Define validation as measurement with no updates -------------------------
def evaluate(model, dataloader):
    model.eval()
    total_loss = 0.0
    with torch.inference_mode():
        for batch_X, batch_y in dataloader:
            batch_X, batch_y = batch_X.to(device), batch_y.to(device)
            loss = loss_function(model(batch_X), batch_y)
            total_loss += loss.item() * len(batch_X)
    return total_loss / len(dataloader.dataset)

# Copy live state to prove evaluation neither updates weights nor writes new gradients.
weights_before_eval = [p.detach().clone() for p in model.parameters()]
grads_before_eval = [None if p.grad is None else p.grad.clone() for p in model.parameters()]
evaluation_probe_loss = evaluate(model, val_loader)
assert all(torch.equal(before, after) for before, after in zip(weights_before_eval, model.parameters()))
assert all(
    (before is None and after.grad is None)
    or (before is not None and after.grad is not None and torch.equal(before, after.grad))
    for before, after in zip(grads_before_eval, model.parameters())
)
print(f"Validation probe loss: {evaluation_probe_loss:.4f}; weights and gradient buffers unchanged.")


### Two returns, different measurement conditions

- Training MSE summarizes batches while weights change and dropout is active.
- Validation MSE checks fixed end-of-epoch weights with dropout off.
- Both average **examples**, including the smaller last batch.

So validation below training isn't automatically a bug. First compare
the conditions under which each number was collected.


### Training can keep improving after useful progress stops

An **early-stopping** observer remembers the lowest validation loss and
how many consecutive checks haven't beaten it. Improvement resets the
counter; reaching patience says stop.

For example, a best value of 0.8 followed by 0.81 and 0.82 means two bad
checks, not two steps of training. Strictly lower counts as improvement
here, so a tie also consumes patience.

The next class stores those decision variables and returns two booleans:
`improved` and `should_stop`. It doesn't train, evaluate, or save anything.
The scheduler has its **own** patience; stopping has patience 10.
Reducing the rate and ending the run are separate decisions.


In [ ]:
# -- Define one small early-stopping state machine ----------------------------
class EarlyStopping:
    def __init__(self, patience):
        self.patience = patience
        self.best = float("inf")
        self.bad_epochs = 0

    def update(self, value):
        improved = value < self.best
        if improved:
            self.best = value
            self.bad_epochs = 0
        else:
            self.bad_epochs += 1
        return improved, self.bad_epochs >= self.patience

early_stopping = EarlyStopping(config.patience)
print(f"Early stopping patience: {early_stopping.patience} validation checks")


## 5 - Keep the winner separately from the latest state

![The best-model file includes weights and preprocessing for inference; a separate snapshot records selected process state but lacks several recovery inputs.](../images/05-two-checkpoint-jobs.png)

Prepare two paths before training:

- `best_model.pt`: overwrite only when validation improves; include
  the model and its input/target preprocessing.
- `resume_training.pt`: later write selected state from the latest
  completed epoch, which may not be the winning epoch.

`history` starts empty so every plotted point must come from a real call.
Its rate entry will record the rate **after** that epoch's scheduler
decision, usually used for the next epoch.

> **Visual to add — The winning lap isn't always the final lap**
> Show an unnumbered sequence of actual-run placeholders labelled
> "validation check", with a movable marker "best so far" and a separate
> "latest completed" marker. Do not draw fake loss values. Route the best
> marker to `best_model.pt` with training rulers, and the latest marker
> to `resume_training.pt` with optimizer/scheduler/counters.
> Suggested caption: "Selection asks what to use; progress asks where we stopped."

The next cell only creates paths and empty records. The loop must actually
write a file before a reload can tell us anything.


In [ ]:
# -- Prepare checkpoint paths and history -------------------------------------
artifact_dir = LESSON_DIR / "artifacts"
artifact_dir.mkdir(exist_ok=True)
best_path = artifact_dir / "best_model.pt"
resume_path = artifact_dir / "resume_training.pt"
history = {"train": [], "validation": [], "learning_rate": []}
best_epoch = -1
print(f"Checkpoint directory: {artifact_dir.resolve()}")


### Read the assembled loop in event order

1. Train one epoch; weights and optimizer state change.
2. Evaluate validation; weights and gradient buffers stay unchanged.
3. Let the scheduler observe validation and choose the next rate.
4. Update best-loss and stopping counters.
5. Record measured losses/rate; save a winner if improved; stop if required.

Each winner file contains weights **and the train-derived input and target
rulers**. A later worse epoch must not overwrite it. The in-memory model at
the end can therefore differ from the best file.

The block can run all 80 allowed epochs or stop earlier. Use its printed
epoch count and best epoch; don't assume patience must be exhausted.


In [ ]:
# -- Assemble the visible training lifecycle ----------------------------------
for epoch in range(config.max_epochs):
    train_loss = train_one_epoch(model, train_loader, optimizer, scaler)
    validation_loss = evaluate(model, val_loader)
    scheduler.step(validation_loss)
    improved, should_stop = early_stopping.update(validation_loss)
    history["train"].append(train_loss)
    history["validation"].append(validation_loss)
    history["learning_rate"].append(optimizer.param_groups[0]["lr"])
    if improved:
        best_epoch = epoch
        torch.save({
            "model_state": model.state_dict(),
            "train_mean": train_mean,
            "train_std": train_std,
            "target_mean": y_mean,
            "target_std": y_std,
            "config": asdict(config),
        }, best_path)
    if should_stop:
        break

print(f"Ran {len(history['train'])} epochs; best validation epoch={best_epoch + 1}")
print(f"Validation loss: {history['validation'][0]:.4f} -> {min(history['validation']):.4f}")


### Pack the latest process state too

The next save includes the latest model, optimizer, scheduler, epoch,
early-stopping variables, and configuration. Those counters matter:
restarting them would change when a rate reduction or stop happens.

This teaching snapshot omits loader/dropout random states, scaler state,
and preprocessing statistics, and we don't restore a continuation from it.
The historical name `resume_training.pt` doesn't prove recovery.
Matching the same next update would need a separate restore-and-compare
experiment with every relevant input and state.

Keep that limit in mind once; the next plot and key check do smaller,
useful jobs.


In [ ]:
# -- Save training state separately from the best inference state ------------
torch.save({
    "epoch": epoch,
    "model_state": model.state_dict(),
    "optimizer_state": optimizer.state_dict(),
    "scheduler_state": scheduler.state_dict(),
    "early_stopping": vars(early_stopping),
    "config": asdict(config),
}, resume_path)
print(f"Best-model checkpoint: {best_path}")
print(f"Training-state snapshot: {resume_path}")


### Put the recorded decisions on a timeline

The first panel plots normalized MSE under the two measurement conditions:
training during updates, validation after each epoch in eval mode.
Look for where validation reaches its minimum, not just the final point.

The second panel steps through rates after scheduler decisions.
A flat trace means no reduction occurred. The printed reduction count
comes from consecutive recorded rates; the chart doesn't invent a plateau.

Plotting changes no weights. It can explain **when** progress changed,
but it can't check that a saved model can rebuild predictions in another
object. That's the next job.


In [ ]:
# -- Plot only recorded history ------------------------------------------------
epochs = range(1, len(history["train"]) + 1)
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(epochs, history["train"], label="train")
axes[0].plot(epochs, history["validation"], label="validation")
axes[0].set(xlabel="epoch", ylabel="normalized MSE", title="Recorded losses")
axes[0].legend()
axes[1].step(epochs, history["learning_rate"], where="post")
axes[1].set(xlabel="epoch", ylabel="learning rate", title="Scheduler decisions")
for axis in axes:
    axis.grid(alpha=0.3)
plt.show()
rate_reductions = sum(
    later < earlier
    for earlier, later in zip(history["learning_rate"], history["learning_rate"][1:])
)
print(f"Recorded learning-rate reductions: {rate_reductions}")
print("These are measured curves; a flat rate trace means no reduction occurred.")


## 6 - Open the winner and put fuel back in fuel units

The next block loads `best_model.pt` into a fresh model. It reconstructs
test features from **raw test rows and the saved rulers**, not from a live
normalized array that might hide a missing file field.

The model returns `[36,1]` normalized predictions. Multiply by the saved
target scale and add its saved mean to return to synthetic fuel units.
Mean absolute error then averages `abs(prediction - target)` over the
36 test rows. Unlike MSE, MAE has the target's original units.

The printed MAE is the test measurement after selection. Repeatedly tuning
settings based on it would turn those rows into validation too.
This generated fuel task isn't evidence about a real engine.


In [ ]:
# -- Reload best state into a fresh model and verify test inference -----------
best_checkpoint = torch.load(best_path, map_location=device, weights_only=True)
best_model = FuelEfficiencyRegressor().to(device)
best_model.load_state_dict(best_checkpoint["model_state"])
best_model.eval()
with torch.inference_mode():
    # Raw (36, 2) with saved (2,) statistics -> (36, 2): reconstruct each sensor ruler.
    saved_X_test = (
        X_test_raw.to(device) - best_checkpoint["train_mean"]
    ) / best_checkpoint["train_std"]
    test_prediction_n = best_model(saved_X_test).cpu()
# (36 test rows, 1 prediction) * (1,) + (1,) -> (36, 1): reuse the
# saved target scale/mean to turn every prediction back into fuel units.
test_prediction = test_prediction_n * best_checkpoint["target_std"].cpu() + best_checkpoint["target_mean"].cpu()
# Absolute fuel misses (36, 1) -> scalar mean: report original-unit MAE.
test_mae = (test_prediction - y_test).abs().mean().item()
print(f"Test MAE in original fuel units: {test_mae:.4f}")
print(f"Prediction shape: {tuple(test_prediction.shape)}; target shape: {tuple(y_test.shape)}")


### Check what was written, not what wasn't tested

Load the separate latest-state snapshot and compare its keys with the
writer's declared fields. An empty missing-key list means that save wrote
what it said it would.

It doesn't load an optimizer into a live run or compare the next batch and
step with uninterrupted learning. The omitted recovery inputs stay omitted
even when every key matches.


In [ ]:
# -- Check snapshot keys, not interrupted-training recovery ---------
resume_checkpoint = torch.load(resume_path, map_location=device, weights_only=True)
required = {"epoch", "model_state", "optimizer_state", "scheduler_state", "early_stopping", "config"}
missing = required - set(resume_checkpoint)
print(f"Missing declared snapshot fields: {sorted(missing)}")
assert not missing


The two file checks answered different questions: the winner supplied
preprocessing and weights for test inference; the latest-state file
contained its declared fields. Neither file check measured crash recovery.


## Optional translations - use automation after the jobs are clear

Lightning moves these responsibilities into hooks and callbacks.
`torch.compile` may optimize a correct computation. Neither decides which
rows are allowed to fit preprocessing, which mode validation needs, or what
your checkpoint must contain. These are translations to read about, not
executed comparisons here.

### Your turn - isolate patience from training noise

Use the deliberately supplied sequence `[1.0,0.8,0.81,0.82,0.83,0.84]`.
It improves at check 2, then has four non-improving checks.
With `YOUR_PATIENCE=4`, stop at check **6**.

These are test inputs for a decision rule, not manufactured training
measurements. Change only patience. A larger value may yield `None`
because this sequence ends before patience does. The next cell compares
the observed stopping point with the derived expectation without retraining.


In [ ]:
# -- CHANGE THIS: inspect the stopping rule without retraining -----------------
YOUR_PATIENCE = 4
assert YOUR_PATIENCE > 0
simulated = EarlyStopping(YOUR_PATIENCE)
sequence = [1.0, 0.8, 0.81, 0.82, 0.83, 0.84]
stop_at = None
for index, value in enumerate(sequence, start=1):
    _, stopped = simulated.update(value)
    if stopped:
        stop_at = index
        break
print(f"With patience={YOUR_PATIENCE}, the simulated run stops at check {stop_at}.")
expected_stop = 2 + YOUR_PATIENCE if 2 + YOUR_PATIENCE <= len(sequence) else None
assert stop_at == expected_stop


## The short version

- Training rows choose preprocessing rulers. Validation rows choose
  settings and a winner. Test rows report only after those choices finish.
- Parameters, activations, gradients, optimizer memory, fitted statistics,
  and stopping counters are different state with different lifetimes.
- Dropout masks activations in train mode, not stored weights. Eval mode
  and inference mode solve separate problems.
- Report per-example loss. The 12-row remainder doesn't deserve a
  24-row vote.
- Scheduling changes future rate; stopping ends practice; saving the best
  model chooses which earlier state to use.
- Reload preprocessing with weights, then undo target normalization before
  reporting original-unit error.
- The best file is inference-tested. The latest snapshot is key-checked,
  not recovery-tested.

| Coverage | Where we stopped |
|---|---|
| Built and measured | Split and ruler difference; normalized shapes; loaders; repeated-mode probe; training/evaluation loops; evaluation side-effect audit; scheduler history; stopping; winner save/reload; latest-snapshot keys; test MAE |
| Explained | CUDA-only AMP/scaling path; selection versus progress state; Lightning/compile mapping; missing recovery inputs |
| Later | Exact interrupted-run recovery, clipping, accumulation, trackers, distributed training and checkpointing need dedicated experiments |

**Remember:** automation can shorten the loop, but it can't decide which
evidence you were allowed to use.


**Carry this into the next task:** a fuel row has two measurements, but its prediction does not remember earlier rows. [Sequence Memory](../../03-language-model-foundations/01-sequence-memory/README.md) keeps the same training responsibilities and asks how a state can carry earlier information through ordered steps. The fuel checkpoint is evidence for this toy model, not a language-model checkpoint to load there.